# MLflow - data selection

This notebook pulls runs and graph artifacts from MLflow, selects the relevant runs (drops parent runs and keeps the actual folds/trials), and saves local snapshots in `RESULTS_DIR`. Evaluation notebooks load these files without connecting to MLflow.

In [ ]:
import json
import os

import dotenv
import mlflow
import pandas as pd
import seaborn as sns
from mlflow.environment_variables import MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR

dotenv.load_dotenv(override=True)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
RESULTS_DIR = os.getenv("RESULTS_DIR")

In [ ]:
if MLFLOW_TRACKING_URI:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

mlflow.get_tracking_uri()

In [ ]:
experiments = mlflow.search_experiments()
df_experiments = pd.DataFrame(
    [
        {
            "experiment_id": exp.experiment_id,
            "name": exp.name,
            "artifact_location": exp.artifact_location,
            "lifecycle_stage": exp.lifecycle_stage,
        }
        for exp in experiments
    ]
)

df_experiments.sort_values("name").reset_index(drop=True)

# Best result per configuration

We load only the experiments in `EXPERIMENTS`, then for each combination
(experiment, model class, graph, sample size, threshold) we take the run with
the best AUC.


In [ ]:
EXPERIMENTS = [
    "pairwise_2",
    "higgs_2",
    "medical_9",
]

# EXPERIMENTS = df_experiments["name"].tolist()

EXPERIMENTS = list(set(EXPERIMENTS) - set(["Default"]))

# Keep only experiments that actually exist in the tracking store.
available_experiments = set(df_experiments["name"])
EXPERIMENTS = [name for name in EXPERIMENTS if name in available_experiments]

selected_runs_df0 = mlflow.search_runs(
    experiment_names=EXPERIMENTS,
    filter_string="",
    order_by=["start_time DESC"],
    max_results=50000,
)

In [ ]:
EXPERIMENTS

In [ ]:
selected_runs_df = selected_runs_df0.copy()

# Drop the per-trial parent runs (they only hold aggregated means) so each
# config is represented by its actual fold / xgboost runs.
if "tags.run_type" in selected_runs_df.columns:
    selected_runs_df = selected_runs_df[selected_runs_df["tags.run_type"] != "parent"]

# Attach the experiment name for grouping/readability.
selected_runs_df = selected_runs_df.merge(
    df_experiments[["experiment_id", "name"]].rename(
        columns={"name": "experiment_name"}
    ),
    on="experiment_id",
    how="left",
)

len(selected_runs_df)

# Deduplication

In [ ]:
MAIN_DATA_SEED = "42"
CONFIG_COLUMNS = [
    "experiment_name",
    "tags.model_cls",
    "tags.graph_name",
    "tags.threshold",
    "tags.n_samples",
    "tags.data_seed",
    "tags.n_edges",
    "tags.cov",
    "tags.train_size",
]
CONFIG_COLUMNS = [column for column in CONFIG_COLUMNS if column in selected_runs_df]

run_type = selected_runs_df["tags.run_type"]
is_final_refit = run_type.eq("final_refit")
final_refit_runs = selected_runs_df[
    is_final_refit & selected_runs_df["status"].eq("FINISHED")
].copy()
tuning_runs = selected_runs_df[~is_final_refit].copy()

trial_id = tuning_runs["tags.trial_id"].astype("string").fillna("missing")
parent_run_id = tuning_runs["tags.mlflow.parentRunId"].astype("string")
tuning_runs["_trial"] = parent_run_id.fillna("trial-" + trial_id)

valid_run = (
    tuning_runs["status"].eq("FINISHED")
    & pd.to_numeric(tuning_runs["metrics.val/auc"], errors="coerce").notna()
)
trial_columns = CONFIG_COLUMNS + ["_trial"]
valid_trial = valid_run.groupby(
    [tuning_runs[column] for column in trial_columns],
    dropna=False,
).transform("all")
valid_runs = tuning_runs[valid_trial].copy()

other_seed = valid_runs["tags.data_seed"].notna() & valid_runs["tags.data_seed"].astype(
    "string"
).ne(MAIN_DATA_SEED)
dedup_columns = CONFIG_COLUMNS + ["tags.fold_id"]
selected_tuning_runs = pd.concat(
    [
        valid_runs[~other_seed],
        valid_runs[other_seed]
        .sort_values("start_time")
        .drop_duplicates(dedup_columns, keep="first"),
    ],
    ignore_index=True,
).drop(columns="_trial")

selected_runs_df = pd.concat(
    [selected_tuning_runs, final_refit_runs],
    ignore_index=True,
)

selected_runs_df[["tags.data_seed", "tags.trial_id"]].value_counts().sort_index()

# Report

In [ ]:
train_size = pd.to_numeric(selected_runs_df.get("params.train_size"), errors="coerce")
is_medical = selected_runs_df["experiment_name"].eq("medical_9")
selected_runs_df = selected_runs_df[
    is_medical | train_size.isna() | train_size.le(10_000)
]

In [ ]:
experiment_name_map = {
    "medical_9": "medical",
    "higgs_2": "higgs",
    "pairwise_2": "pairwise",
}

selected_runs_df["experiment_name"] = (
    selected_runs_df["experiment_name"]
    .map(experiment_name_map)
    .fillna(selected_runs_df["experiment_name"])
)

In [ ]:
selected_runs_df[["experiment_name", "tags.cov", "tags.train_size"]].value_counts()

In [ ]:
selected_runs_df[["experiment_name"]].value_counts()

# Progress control

In [ ]:
selected_runs_df["params.n_edges"] = (
    selected_runs_df["params.n_edges"].fillna(-1).astype(int)
)

In [ ]:
df_tmp = selected_runs_df[selected_runs_df["experiment_name"] == "pairwise"].copy()
df_tmp = df_tmp[df_tmp["tags.model_cls"] != "XGBoost"]

In [ ]:
df_tmp[
    ["experiment_name", "tags.graph_permute_seed", "tags.cov", "tags.train_size"]
].value_counts().sort_index().to_frame()

In [ ]:
df_tmp[
    ["tags.model_cls", "tags.cov", "tags.train_size", "tags.n_edges"]
].value_counts().sort_index().to_frame()

In [ ]:
time_s = df_tmp["start_time"]
time_s = time_s[time_s > pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=2)]  # last week

In [ ]:
# Full half-hour windows.
time_s = time_s.dt.floor("30min")
time_s.sort_values()

In [ ]:
fig = sns.histplot(time_s, bins=len(time_s.value_counts().sort_index()))
fig.xaxis.set_tick_params(rotation=45)

In [ ]:
time_s.value_counts().sort_index()

In [ ]:
selected_runs_df["experiment_name"].value_counts().sort_index().to_frame()

# Save to CSV

We save the selected runs to a CSV file in `RESULTS_DIR`. This file is loaded by
`mlflow_results_evaluate_synthetic.ipynb` for further analysis.


In [ ]:
os.makedirs(RESULTS_DIR, exist_ok=True)
csv_path = os.path.join(RESULTS_DIR, "mlflow_selected_runs_df.csv")
schema_path = os.path.join(RESULTS_DIR, "mlflow_selected_runs_schema.json")

parameter_columns = [
    column.split(".", 1)[1]
    for column in selected_runs_df.columns
    if column.startswith("params.")
]
metric_columns = [
    column.split(".", 1)[1]
    for column in selected_runs_df.columns
    if column.startswith("metrics.")
]

# Keep selected_runs_df unchanged for the artifact-export cells below.
column_order = sorted(
    range(len(selected_runs_df.columns)),
    key=lambda index: "." in selected_runs_df.columns[index],
)
export_df = selected_runs_df.iloc[:, column_order].copy()
export_df.columns = [
    column.split(".", 1)[1] if "." in column else column for column in export_df.columns
]
export_df = export_df.T.groupby(level=0, sort=False).first().T

export_df.to_csv(csv_path, index=False)
with open(schema_path, "w", encoding="utf-8") as schema_file:
    json.dump(
        {
            "parameter_columns": list(dict.fromkeys(parameter_columns)),
            "metric_columns": list(dict.fromkeys(metric_columns)),
        },
        schema_file,
        indent=2,
    )

print(f"Saved {len(export_df)} runs to {csv_path}")
print(f"Saved column schema to {schema_path}")

# Save graph artifacts

Medical interaction-graph artifacts are downloaded once and saved to `mlflow_graph_artifacts.json`. Evaluation notebooks read this local snapshot instead of connecting to MLflow.

In [ ]:
CV_GRAPH_ARTIFACT_PATH = "graph/estimated_graph.json"
FINAL_GRAPH_ARTIFACT_PATH = "final_graph.json"
GRAPH_ARTIFACTS_FILENAME = "mlflow_graph_artifacts.json"

graph_name_columns = [
    column
    for column in ["tags.graph_name", "params.graph_name"]
    if column in selected_runs_df
]
if not graph_name_columns:
    raise ValueError("No graph name column found in selected MLflow runs")

graph_names = selected_runs_df[graph_name_columns].bfill(axis=1).iloc[:, 0]
run_types = selected_runs_df.get(
    "tags.run_type",
    pd.Series(pd.NA, index=selected_runs_df.index),
)
medical_interaction_runs = selected_runs_df[
    selected_runs_df["experiment_name"].eq("medical")
    & graph_names.str.startswith("ii_graph_", na=False)
].copy()
medical_interaction_runs["artifact_path"] = pd.NA
medical_interaction_runs.loc[
    run_types.loc[medical_interaction_runs.index].eq("fold"),
    "artifact_path",
] = CV_GRAPH_ARTIFACT_PATH
medical_interaction_runs.loc[
    run_types.loc[medical_interaction_runs.index].eq("final_refit"),
    "artifact_path",
] = FINAL_GRAPH_ARTIFACT_PATH
artifact_candidates = medical_interaction_runs.dropna(subset=["artifact_path"])[
    ["run_id", "artifact_path"]
].drop_duplicates()

graph_artifacts = []
artifact_download_errors = []
MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR.set(False)
for run_id, artifact_path in artifact_candidates.itertuples(index=False, name=None):
    try:
        local_path = mlflow.artifacts.download_artifacts(
            run_id=run_id,
            artifact_path=artifact_path,
        )
    except mlflow.exceptions.MlflowException as error:
        artifact_download_errors.append(
            {"run_id": run_id, "artifact_path": artifact_path, "error": str(error)}
        )
        continue

    with open(local_path, encoding="utf-8") as artifact_file:
        payload = json.load(artifact_file)
    edges = sorted(
        {
            tuple(sorted((str(source), str(target))))
            for source, target in payload["edges"]
        }
    )
    if len(edges) != payload["n_edges"]:
        raise ValueError(
            f"Invalid graph artifact for run {run_id}: "
            f"expected {payload['n_edges']} edges, found {len(edges)}"
        )
    graph_artifacts.append(
        {
            "run_id": run_id,
            "artifact_path": artifact_path,
            "n_edges": len(edges),
            "edges": edges,
        }
    )

graph_artifacts_path = os.path.join(RESULTS_DIR, GRAPH_ARTIFACTS_FILENAME)
with open(graph_artifacts_path, "w", encoding="utf-8") as artifact_file:
    json.dump(
        {
            "artifacts": graph_artifacts,
            "download_errors": artifact_download_errors,
        },
        artifact_file,
        ensure_ascii=False,
        indent=2,
    )

print(f"Saved {len(graph_artifacts)} graph artifacts to {graph_artifacts_path}")
if artifact_download_errors:
    print(f"Skipped {len(artifact_download_errors)} unavailable graph artifacts")